In [ ]:
# AIF360 Bias Detection & Mitigation Workshop
# For Directors and Managers: Understanding Ethical AI Toolboxes

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# AIF360 imports
from aif360.datasets import AdultDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.postprocessing import EqOddsPostprocessing

print(" Welcome to the AIF360 Ethical AI Workshop!")
print("=" * 60)

In [ ]:
# =============================================================================
# SECTION 1: BUSINESS CONTEXT - WHY THIS MATTERS TO LEADERSHIP
# =============================================================================

print("\n BUSINESS CONTEXT FOR LEADERSHIP")
print("=" * 40)
print("""
KEY QUESTIONS FOR DIRECTORS/MANAGERS:
• What are the financial/legal risks of biased AI systems?
• How do we measure and communicate bias to stakeholders?
• What tools can our data science teams use to mitigate bias?
• How do we balance fairness with business performance?

This notebook demonstrates AIF360's capabilities using a hiring scenario.
""")

In [ ]:
# =============================================================================
# SECTION 2: DATASET OVERVIEW - WHAT WE'RE WORKING WITH
# =============================================================================

print("\n SCENARIO: AI-POWERED HIRING DECISIONS")
print("=" * 40)

# Load the Adult Dataset (classic ML fairness benchmark)
dataset = AdultDataset()
df = dataset.convert_to_dataframe()[0]

print(f"Dataset size: {len(df):,} job candidates")
print(f"Features: {df.shape[1]} candidate attributes")
print("\n BUSINESS SCENARIO:")
print("We're building an AI system to predict which job candidates")
print("are likely to earn >$50K (proxy for 'high-potential hires')")

# Show key demographics
print("\n CANDIDATE DEMOGRAPHICS:")
print("Gender distribution:")
gender_dist = df['sex'].value_counts()
print(f"  Female: {gender_dist[0]:,} ({gender_dist[0]/len(df)*100:.1f}%)")
print(f"  Male: {gender_dist[1]:,} ({gender_dist[1]/len(df)*100:.1f}%)")

print(f"\nAge range: {df['age'].min()}-{df['age'].max()} years")
print(f"Education levels: {df['education'].nunique()} categories")

In [ ]:
# =============================================================================
# SECTION 3: THE BIAS PROBLEM - MEASURING UNFAIRNESS
# =============================================================================

print("\n MEASURING BIAS: WHAT AIF360 DETECTS")
print("=" * 45)

# Split dataset
train, test = dataset.split([0.7], shuffle=True, seed=123)

# Define protected groups (what we're checking for bias against)
privileged_groups = [{'sex': 1}]    # Male = 1
unprivileged_groups = [{'sex': 0}]  # Female = 0

# Calculate initial bias metrics
metric_orig_train = BinaryLabelDatasetMetric(
    train,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print(" BIAS METRICS IN ORIGINAL DATA:")
print(f"Disparate Impact: {metric_orig_train.disparate_impact():.3f}")
print(f"Statistical Parity Difference: {metric_orig_train.statistical_parity_difference():.3f}")

print("\n WHAT THESE NUMBERS MEAN FOR LEADERSHIP:")
print("• Disparate Impact < 0.8 = Potential legal risk (80% rule)")
print("• Statistical Parity Difference > 0.1 = Significant bias concern")

# Visualize the bias
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Outcome rates by gender
outcome_by_gender = df.groupby('sex')['income-per-year'].mean()
ax1.bar(['Female', 'Male'], outcome_by_gender, color=['#ff7f7f', '#7f7fff'])
ax1.set_title('High-Income Rate by Gender\n(Original Data)')
ax1.set_ylabel('Rate of High-Income (>50K)')
ax1.set_ylim(0, 0.5)
for i, v in enumerate(outcome_by_gender):
    ax1.text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# Add risk assessment
disparate_impact = metric_orig_train.disparate_impact()
risk_level = " HIGH RISK" if disparate_impact < 0.8 else " MEDIUM RISK" if disparate_impact < 0.9 else " LOW RISK"
ax2.text(0.5, 0.7, f'Disparate Impact: {disparate_impact:.3f}', 
         ha='center', va='center', fontsize=14, fontweight='bold')
ax2.text(0.5, 0.5, risk_level, ha='center', va='center', fontsize=16, fontweight='bold')
ax2.text(0.5, 0.3, 'Legal threshold: 0.8', ha='center', va='center', fontsize=12)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_title('Legal Risk Assessment')
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SECTION 4: AIF360'S SOLUTION - BIAS MITIGATION TECHNIQUES
# =============================================================================

print("\n AIF360'S BIAS MITIGATION APPROACH")
print("=" * 42)
print("AIF360 offers 3 types of bias mitigation:")
print("1. PRE-processing: Fix bias in training data")
print("2. IN-processing: Build fairness into model training")
print("3. POST-processing: Adjust model outputs for fairness")
print("\nWe'll demonstrate PRE-processing (Reweighing)...")

# Apply Reweighing (adjusts training weights to reduce bias)
np.random.seed(123)
reweighing = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)
dataset_reweighed = reweighing.fit_transform(train)

# Check bias after reweighing
metric_reweighed = BinaryLabelDatasetMetric(
    dataset_reweighed,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print("\n RESULTS AFTER REWEIGHING:")
print(f"Original Disparate Impact: {metric_orig_train.disparate_impact():.3f}")
print(f"After Reweighing: {metric_reweighed.disparate_impact():.3f}")
print(f"Improvement: {((metric_reweighed.disparate_impact() - metric_orig_train.disparate_impact()) / metric_orig_train.disparate_impact() * 100):+.1f}%")

In [ ]:
# =============================================================================
# SECTION 5: BUSINESS IMPACT ANALYSIS
# =============================================================================

print("\n BUSINESS IMPACT ANALYSIS")
print("=" * 30)

# Train models (original vs. bias-mitigated)
scaler = StandardScaler()

# Original model
X_train_orig = scaler.fit_transform(train.features)
y_train_orig = train.labels.ravel()
model_orig = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=123)
model_orig.fit(X_train_orig, y_train_orig)

# Bias-mitigated model
X_train_fair = scaler.fit_transform(dataset_reweighed.features)
y_train_fair = dataset_reweighed.labels.ravel()
model_fair = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=123)
model_fair.fit(X_train_fair, y_train_fair, sample_weight=dataset_reweighed.instance_weights)

# Test both models
X_test = scaler.transform(test.features)
y_test = test.labels.ravel()

y_pred_orig = model_orig.predict(X_test)
y_pred_fair = model_fair.predict(X_test)

# Calculate business metrics
accuracy_orig = accuracy_score(y_test, y_pred_orig)
accuracy_fair = accuracy_score(y_test, y_pred_fair)

# Create business comparison dashboard
print(" EXECUTIVE DASHBOARD: MODEL COMPARISON")
print("=" * 45)

# Performance metrics
print(" BUSINESS PERFORMANCE:")
print(f"Original Model Accuracy: {accuracy_orig:.1%}")
print(f"Fair Model Accuracy: {accuracy_fair:.1%}")
print(f"Performance Trade-off: {accuracy_fair - accuracy_orig:+.1%}")

# Fairness metrics for test predictions
test_pred_orig = test.copy()
test_pred_orig.labels = y_pred_orig
metric_test_orig = BinaryLabelDatasetMetric(
    test_pred_orig, unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups
)

test_pred_fair = test.copy()
test_pred_fair.labels = y_pred_fair
metric_test_fair = BinaryLabelDatasetMetric(
    test_pred_fair, unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups
)

print("\n FAIRNESS METRICS:")
print(f"Original Model Disparate Impact: {metric_test_orig.disparate_impact():.3f}")
print(f"Fair Model Disparate Impact: {metric_test_fair.disparate_impact():.3f}")

# Risk assessment
orig_risk = " HIGH" if metric_test_orig.disparate_impact() < 0.8 else " MEDIUM" if metric_test_orig.disparate_impact() < 0.9 else " LOW"
fair_risk = " HIGH" if metric_test_fair.disparate_impact() < 0.8 else " MEDIUM" if metric_test_fair.disparate_impact() < 0.9 else " LOW"

print(f"\n LEGAL RISK LEVEL:")
print(f"Original Model: {orig_risk}")
print(f"Fair Model: {fair_risk}")

In [ ]:
# =============================================================================
# SECTION 6: EXECUTIVE SUMMARY & RECOMMENDATIONS
# =============================================================================

print("\n" + "="*60)
print(" EXECUTIVE SUMMARY & RECOMMENDATIONS")
print("="*60)

print("\n WHAT AIF360 PROVIDES YOUR ORGANIZATION:")
print("• Standardized bias measurement across different fairness definitions")
print("• Pre-built algorithms to reduce bias in ML models")
print("• Clear metrics that translate to legal/compliance requirements")
print("• Integration with popular ML frameworks (sklearn, etc.)")

print("\n KEY DECISION POINTS FOR LEADERSHIP:")
print(f"1. Performance vs. Fairness Trade-off: {accuracy_fair - accuracy_orig:+.1%} accuracy change")
print(f"2. Legal Risk Mitigation: {orig_risk} → {fair_risk}")
print("3. Implementation Effort: Medium (requires data science team training)")
print("4. Ongoing Monitoring: Automated bias metrics in production")

print("\n RECOMMENDED NEXT STEPS:")
print("1. Establish bias monitoring KPIs for your ML models")
print("2. Train data science teams on AIF360 implementation")
print("3. Integrate bias testing into ML model validation process")
print("4. Create executive dashboard for ongoing fairness monitoring")

print("\n QUESTIONS FOR DISCUSSION:")
print("• What level of performance trade-off is acceptable for reduced bias?")
print("• How should we communicate fairness metrics to business stakeholders?")
print("• What governance processes need to change to include bias testing?")
print("• How do we balance different fairness definitions for different use cases?")

print("\n" + "="*60)
print(" End of AIF360 Workshop - Ready for Q&A!")
print("="*60)